In [ ]:
!ls

In [ ]:

# CELL 1 — Mount Google Drive & Create Project Directories
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Define base save path
BASE_PATH = '/content/drive/MyDrive/CSET419_Lab11'

# Create all necessary subdirectories
dirs = [
    BASE_PATH,
    f'{BASE_PATH}/component1_reviews',
    f'{BASE_PATH}/component1_reviews/model',
    f'{BASE_PATH}/component1_reviews/outputs',
    f'{BASE_PATH}/component2_recipes',
    f'{BASE_PATH}/component2_recipes/model',
    f'{BASE_PATH}/component2_recipes/outputs',
    f'{BASE_PATH}/reports',
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("✅ Google Drive mounted successfully!")
print(f"📁 Project folder: {BASE_PATH}")
print("\nDirectory structure created:")
for d in dirs:
    print(f"  ✓ {d.replace(BASE_PATH, 'MyDrive/CSET419_Lab11')}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Install Required Libraries
# ═══════════════════════════════════════════════════════════════
!pip install transformers datasets accelerate -q

print("✅ Libraries installed:")
print("  • transformers  — GPT-2 model & tokenizer")
print("  • datasets      — Dataset handling")
print("  • accelerate    — PyTorch training acceleration")
print("  • torch         — PyTorch (pre-installed in Colab)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Imports & Device Configuration
# ═══════════════════════════════════════════════════════════════
import torch
import math
import json
import os
from datetime import datetime

from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
from datasets import Dataset

# ── Reproducibility ──────────────────────────────────────────
set_seed(42)

# ── Device Detection ─────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 50)
print("  CSET419 Lab 11 – Fine-Tuning GPT-2")
print("=" * 50)
print(f"\n🖥️  Device     : {DEVICE.upper()}")
print(f"🔦  PyTorch    : {torch.__version__}")

if DEVICE == "cuda":
    print(f"⚡  GPU        : {torch.cuda.get_device_name(0)}")
    print(f"💾  VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found. Go to Runtime → Change runtime type → T4 GPU")

print(f"\n🌱  Random seed : 42 (set for reproducibility)")
print(f"📅  Timestamp   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — PyTorch Utility Functions
# ═══════════════════════════════════════════════════════════════

def generate_text(model, tokenizer, prompt,
                  max_new_tokens=80, temperature=0.8,
                  top_k=50, top_p=0.92):
    """Generate text using pure PyTorch inference."""
    model.eval()
    model = model.to(DEVICE)

    inputs = tokenizer.encode(prompt, return_tensors='pt').to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            repetition_penalty=1.2,          # Avoid loops
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


def compute_perplexity(trainer):
    """Compute perplexity from eval loss — lower is better."""
    results = trainer.evaluate()
    ppl = math.exp(results["eval_loss"])
    return ppl, results


def tokenize_corpus(corpus, tokenizer, max_length=128):
    """Tokenize a list of strings into a HuggingFace Dataset."""
    dataset = Dataset.from_dict({'text': corpus})
    tokenized = dataset.map(
        lambda x: tokenizer(
            x['text'], truncation=True,
            max_length=max_length, padding='max_length'
        ),
        batched=True, remove_columns=['text']
    )
    return tokenized.train_test_split(test_size=0.15, seed=42)


def save_outputs_to_drive(data, filename, folder):
    """Save a dictionary as JSON to Google Drive."""
    path = f"{folder}/{filename}"
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"  💾 Saved → {path}")
    return path


def build_training_args(output_dir, epochs=15):
    """Build standard TrainingArguments for both components."""
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_steps=50,
        eval_strategy='epoch',
        logging_steps=10,
        save_strategy='no',
        fp16=torch.cuda.is_available(),
        report_to="none",
    )


print("✅ Utility functions defined:")
print("  • generate_text()         — PyTorch inference with sampling")
print("  • compute_perplexity()    — Eval loss → PPL")
print("  • tokenize_corpus()       — Dataset prep & split")
print("  • save_outputs_to_drive() — JSON export to Drive")
print("  • build_training_args()   — Standard hyperparameters")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Component I: Load GPT-2 for Product Reviews
# ═══════════════════════════════════════════════════════════════
print("⏳ Loading GPT-2 model and tokenizer...")

tokenizer_c1 = GPT2Tokenizer.from_pretrained('gpt2')
model_c1     = GPT2LMHeadModel.from_pretrained('gpt2')

# Fix: GPT-2 has no pad token — set it to eos
tokenizer_c1.pad_token                = tokenizer_c1.eos_token
model_c1.config.pad_token_id          = tokenizer_c1.eos_token_id

model_c1 = model_c1.to(DEVICE)

# ── Model stats ──────────────────────────────────────────────
total_params   = sum(p.numel() for p in model_c1.parameters())
trainable_params = sum(p.numel() for p in model_c1.parameters()
                      if p.requires_grad)

print("\n✅ GPT-2 Loaded (Component I — Reviews)")
print(f"  Model          : GPT-2 (Base)")
print(f"  Device         : {DEVICE.upper()}")
print(f"  Total params   : {total_params:,}")
print(f"  Trainable      : {trainable_params:,}")
print(f"  Vocab size     : {tokenizer_c1.vocab_size:,}")
print(f"  Context window : 1,024 tokens")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Component I: Baseline Reviews (Before Fine-Tuning)
# ═══════════════════════════════════════════════════════════════
REVIEW_PROMPTS = [
    'This product is',
    'I bought this phone and',
    'The quality of this item',
    'I highly recommend',
]

print("=" * 60)
print("  BASELINE REVIEWS — Before Fine-Tuning")
print("=" * 60)

baseline_c1 = {}
for i, prompt in enumerate(REVIEW_PROMPTS, 1):
    output = generate_text(model_c1, tokenizer_c1, prompt)
    baseline_c1[prompt] = output
    print(f"\n[{i}] Prompt   : {prompt}")
    print(f"    Generated: {output[:180]}")

# Save baseline to Drive
save_outputs_to_drive(
    {"stage": "baseline", "component": 1, "outputs": baseline_c1},
    "c1_baseline_reviews.json",
    f"{BASE_PATH}/component1_reviews/outputs"
)
print("\n📌 Observation: Baseline outputs are generic — news/Wikipedia style text")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Component I: Dataset Preparation & Fine-Tuning
# ═══════════════════════════════════════════════════════════════

REVIEW_CORPUS = [
    'this phone has an amazing battery life and the camera quality is outstanding for the price.',
    'i bought this laptop for college and it handles all my assignments and coding projects perfectly.',
    'the sound quality of these headphones is incredible with deep bass and clear vocals.',
    'this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.',
    'great wireless earbuds with noise cancellation that blocks out all background sound.',
    'the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.',
    'this portable charger saved me during travel and it charges my phone three times on a single charge.',
    'the tablet screen is bright and colorful which makes watching movies a great experience.',
    'i love this fitness tracker because it motivates me to reach my daily exercise goals.',
    'this bluetooth speaker is compact but delivers surprisingly loud and clear audio.',
    'the delivery was fast and the product was packed securely with no damage at all.',
    'excellent value for money and the build quality feels premium despite the affordable price.',
    'the customer service team was very helpful when i had questions about the product features.',
    'this camera takes stunning photos in low light and the video recording quality is very smooth.',
    'i have been using this product for three months and it still works perfectly like day one.',
    'the design is sleek and modern and it looks great on my desk next to my other gadgets.',
    'easy to set up right out of the box and the instructions were clear and simple to follow.',
    'highly recommend this product to anyone looking for quality and reliability at a fair price.',
    'the software updates keep adding new features which makes this purchase even more worthwhile.',
    'best purchase i made this year and i would definitely buy from this brand again.',
]

# ── Tokenize ─────────────────────────────────────────────────
print("⏳ Tokenizing review corpus...")
split_c1 = tokenize_corpus(REVIEW_CORPUS, tokenizer_c1)
print(f"  Train samples : {len(split_c1['train'])}")
print(f"  Eval  samples : {len(split_c1['test'])}")

# ── Data Collator ────────────────────────────────────────────
collator_c1 = DataCollatorForLanguageModeling(
    tokenizer=tokenizer_c1, mlm=False   # Causal LM, not masked
)

# ── Training Arguments ───────────────────────────────────────
args_c1 = build_training_args(
    output_dir=f'/content/gpt2-reviews',
    epochs=15
)

# ── Trainer ──────────────────────────────────────────────────
trainer_c1 = Trainer(
    model=model_c1,
    args=args_c1,
    train_dataset=split_c1['train'],
    eval_dataset=split_c1['test'],
    data_collator=collator_c1,
)

print("\n🚀 Starting fine-tuning (15 epochs)...")
print("   Hyperparameters:")
print("   • Epochs    : 15")
print("   • Batch     : 4")
print("   • LR        : 5e-5")
print("   • Warmup    : 50 steps")
print("   • FP16      : True (GPU)\n")

trainer_c1.train()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Component I: Evaluate, Compare & Save
# ═══════════════════════════════════════════════════════════════

# ── Perplexity ───────────────────────────────────────────────
ppl_c1, eval_res_c1 = compute_perplexity(trainer_c1)
print(f"📊 Perplexity (Component I): {ppl_c1:.2f}")
print(f"   (Lower = better domain fit. GPT-2 baseline PPL ~120–200 on this data)")

# ── Compare Baseline vs Fine-Tuned ───────────────────────────
print("\n" + "=" * 65)
print("  COMPARISON: Baseline vs Fine-Tuned — Product Reviews")
print("=" * 65)

comparison_c1 = []
for prompt in REVIEW_PROMPTS:
    ft_out = generate_text(model_c1, tokenizer_c1, prompt)
    rec = {
        "prompt":    prompt,
        "baseline":  baseline_c1[prompt],
        "finetuned": ft_out
    }
    comparison_c1.append(rec)

    print(f"\n📌 Prompt   : {prompt}")
    print(f"  ❌ Baseline  : {baseline_c1[prompt][:140]}")
    print(f"  ✅ Fine-Tuned: {ft_out[:140]}")

# ── Save model to Drive ──────────────────────────────────────
model_save_path_c1 = f"{BASE_PATH}/component1_reviews/model"
print("\n\n💾 Saving fine-tuned model to Drive...")
model_c1.save_pretrained(model_save_path_c1)
tokenizer_c1.save_pretrained(model_save_path_c1)
print(f"  ✅ Model saved → {model_save_path_c1}")

# ── Save outputs JSON ────────────────────────────────────────
save_outputs_to_drive(
    {
        "perplexity": round(ppl_c1, 2),
        "eval_loss":  round(eval_res_c1["eval_loss"], 4),
        "comparisons": comparison_c1
    },
    "c1_finetuned_reviews.json",
    f"{BASE_PATH}/component1_reviews/outputs"
)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — Component II: Load FRESH GPT-2 (not the review model!)
# ═══════════════════════════════════════════════════════════════
print("⏳ Loading a FRESH GPT-2 for recipe generation...")
print("   (This is a clean pre-trained model, not the review one)\n")

tokenizer_c2 = GPT2Tokenizer.from_pretrained('gpt2')
model_c2     = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer_c2.pad_token       = tokenizer_c2.eos_token
model_c2.config.pad_token_id = tokenizer_c2.eos_token_id

model_c2 = model_c2.to(DEVICE)

# Verify models are different objects in memory
print(f"✅ Fresh GPT-2 loaded for Component II")
print(f"   model_c1 id: {id(model_c1)}")
print(f"   model_c2 id: {id(model_c2)}  ← different object ✓")
print(f"   Device     : {next(model_c2.parameters()).device}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — Component II: Baseline Recipes (Before Fine-Tuning)
# ═══════════════════════════════════════════════════════════════
RECIPE_PROMPTS = [
    'To make butter chicken',
    'For pasta carbonara',
    'To prepare a chocolate cake',
    'The first step in making stir fry is',
]

print("=" * 60)
print("  BASELINE RECIPES — Before Fine-Tuning")
print("=" * 60)

baseline_c2 = {}
for i, prompt in enumerate(RECIPE_PROMPTS, 1):
    output = generate_text(model_c2, tokenizer_c2, prompt)
    baseline_c2[prompt] = output
    print(f"\n[{i}] Prompt   : {prompt}")
    print(f"    Generated: {output[:180]}")

save_outputs_to_drive(
    {"stage": "baseline", "component": 2, "outputs": baseline_c2},
    "c2_baseline_recipes.json",
    f"{BASE_PATH}/component2_recipes/outputs"
)
print("\n📌 Observation: Baseline generates off-topic text, not cooking steps")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 11 — Component II: Dataset Preparation & Fine-Tuning
# ═══════════════════════════════════════════════════════════════

RECIPE_CORPUS = [
    'to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one hour.',
    'heat butter in a pan and fry onions until golden brown then add ginger garlic paste and cook for two minutes.',
    'add tomato puree and cook on low heat for ten minutes until the oil separates from the masala.',
    'add the marinated chicken and cook on medium heat for fifteen minutes until fully cooked.',
    'finish with fresh cream and kasuri methi and serve hot with naan or steamed rice.',
    'for pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water.',
    'fry diced pancetta in olive oil until crispy and set aside.',
    'whisk together egg yolks parmesan cheese and black pepper in a bowl.',
    'toss the hot pasta with pancetta and remove from heat then quickly stir in the egg mixture.',
    'the residual heat will cook the eggs into a creamy sauce and serve immediately with extra parmesan.',
    'to prepare vegetable stir fry heat sesame oil in a wok on high heat.',
    'add sliced bell peppers broccoli florets and snap peas and toss for three minutes.',
    'pour in soy sauce oyster sauce and a pinch of sugar and stir well.',
    'add minced garlic and ginger and cook for one more minute until fragrant.',
    'serve the stir fry over steamed jasmine rice and garnish with sesame seeds.',
    'for chocolate chip cookies cream together butter and sugar until light and fluffy.',
    'beat in eggs one at a time then add vanilla extract and mix well.',
    'fold in flour baking soda and salt then gently stir in chocolate chips.',
    'scoop dough onto a baking sheet and bake at 180 degrees for twelve minutes until golden.',
    'let cookies cool on the tray for five minutes before transferring to a wire rack.',
]

# ── Tokenize ─────────────────────────────────────────────────
print("⏳ Tokenizing recipe corpus...")
split_c2 = tokenize_corpus(RECIPE_CORPUS, tokenizer_c2)
print(f"  Train samples : {len(split_c2['train'])}")
print(f"  Eval  samples : {len(split_c2['test'])}")

collator_c2 = DataCollatorForLanguageModeling(tokenizer=tokenizer_c2, mlm=False)

args_c2 = build_training_args(
    output_dir='/content/gpt2-recipes',
    epochs=15
)

trainer_c2 = Trainer(
    model=model_c2,
    args=args_c2,
    train_dataset=split_c2['train'],
    eval_dataset=split_c2['test'],
    data_collator=collator_c2,
)

print("\n🚀 Starting fine-tuning (Component II — Recipes)...")
trainer_c2.train()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 12 — Component II: Evaluate, Compare & Save
# ═══════════════════════════════════════════════════════════════

ppl_c2, eval_res_c2 = compute_perplexity(trainer_c2)
print(f"📊 Perplexity (Component II): {ppl_c2:.2f}")

print("\n" + "=" * 65)
print("  COMPARISON: Baseline vs Fine-Tuned — Recipe Instructions")
print("=" * 65)

comparison_c2 = []
for prompt in RECIPE_PROMPTS:
    ft_out = generate_text(model_c2, tokenizer_c2, prompt)
    comparison_c2.append({
        "prompt":    prompt,
        "baseline":  baseline_c2[prompt],
        "finetuned": ft_out
    })
    print(f"\n📌 Prompt   : {prompt}")
    print(f"  ❌ Baseline  : {baseline_c2[prompt][:140]}")
    print(f"  ✅ Fine-Tuned: {ft_out[:140]}")

# Save model
model_save_path_c2 = f"{BASE_PATH}/component2_recipes/model"
model_c2.save_pretrained(model_save_path_c2)
tokenizer_c2.save_pretrained(model_save_path_c2)
print(f"\n✅ Recipe model saved → {model_save_path_c2}")

save_outputs_to_drive(
    {
        "perplexity":  round(ppl_c2, 2),
        "eval_loss":   round(eval_res_c2["eval_loss"], 4),
        "comparisons": comparison_c2
    },
    "c2_finetuned_recipes.json",
    f"{BASE_PATH}/component2_recipes/outputs"
)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 13 — Final Summary Report & Drive Export
# ═══════════════════════════════════════════════════════════════

report = {
    "lab": "CSET419 Lab 11 — Fine-Tuning GPT-2",
    "timestamp": datetime.now().isoformat(),
    "device": DEVICE.upper(),
    "framework": "PyTorch + Hugging Face Transformers",
    "component_1": {
        "task":            "Product Review Generator (E-Commerce)",
        "model":           "GPT-2 (124M params)",
        "dataset_size":    20,
        "train_epochs":    15,
        "learning_rate":  5e-5,
        "perplexity":      round(ppl_c1, 2),
        "model_saved_at": model_save_path_c1,
        "observation":    "Fine-tuned model generates e-commerce review style text",
    },
    "component_2": {
        "task":            "Recipe Instruction Generator (Food-Tech)",
        "model":           "GPT-2 (124M params)",
        "dataset_size":    20,
        "train_epochs":    15,
        "learning_rate":  5e-5,
        "perplexity":      round(ppl_c2, 2),
        "model_saved_at": model_save_path_c2,
        "observation":    "Fine-tuned model generates step-by-step cooking instructions",
    },
}

# Save report
report_path = save_outputs_to_drive(
    report, "lab11_final_report.json",
    f"{BASE_PATH}/reports"
)

# ── Pretty Print Summary ─────────────────────────────────────
print("\n" + "█" * 60)
print("  LAB 11 COMPLETE — FINAL SUMMARY")
print("█" * 60)
print(f"""
  ┌─────────────────────────────────────────────────────┐
  │  COMPONENT I  :  Product Review Generator           │
  │  Perplexity   :  {ppl_c1:.2f}  (lower = better)          │
  │  Result       :  ✅ Review-style text generated      │
  ├─────────────────────────────────────────────────────┤
  │  COMPONENT II :  Recipe Instruction Generator       │
  │  Perplexity   :  {ppl_c2:.2f}  (lower = better)          │
  │  Result       :  ✅ Step-by-step cooking generated   │
  ├─────────────────────────────────────────────────────┤
  │  Drive Folder : MyDrive/CSET419_Lab11/              │
  │  ├── component1_reviews/model/   (saved model)      │
  │  ├── component1_reviews/outputs/ (JSON results)     │
  │  ├── component2_recipes/model/   (saved model)      │
  │  ├── component2_recipes/outputs/ (JSON results)     │
  │  └── reports/lab11_final_report.json                │
  └─────────────────────────────────────────────────────┘
""")
print("🎉 All outputs successfully saved to Google Drive!")